# 01 — Monte Carlo Pit Stop Simulation
### F1 Pit Stop Optimizer | Mühendislik Seviyesi Analiz

**Bu notebook'ta:**
- Log-Normal dağılım seçiminin fiziksel gerekçesi
- Kritik Yol Yöntemi (CPM) ile bottleneck tespiti
- Monte Carlo engine'in adım adım uygulaması
- Ekip profilleri karşılaştırması
- Sensitivity analizi: hangi görev süreyi en çok etkiliyor?

---


## 1. Neden Log-Normal?

Pit stop görev süreleri için **normal dağılım yanlış** bir seçimdir:
- Normal dağılım negatif değerler üretir (fiziksel olarak imkansız)
- Pit stop hataları sağa çarpık bir kuyruk oluşturur (uzun tail)
- Log-Normal her zaman pozitif, sağa çarpık → gerçekçi

### Dönüşüm formülleri

Gerçek zamandaki $\mu$ ve $\sigma$ parametrelerinden Log-Normal parametrelerine:

$$\sigma_{\ln} = \sqrt{\ln\left(1 + \left(\frac{\sigma}{\mu}\right)^2\right)}$$

$$\mu_{\ln} = \ln(\mu) - \frac{1}{2}\sigma_{\ln}^2$$

Özelliği: $E[X] = e^{\mu_{\ln} + \sigma_{\ln}^2/2} = \mu$ ✓


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import lognorm, norm

# Log-Normal vs Normal karşılaştırması
mu_real, sigma_real = 0.48, 0.06  # gunman task (saniye)

# Log-Normal parametreleri
cv2 = (sigma_real / mu_real) ** 2
sigma_ln = np.sqrt(np.log(1 + cv2))
mu_ln = np.log(mu_real) - 0.5 * np.log(1 + cv2)

x = np.linspace(0, 1.2, 500)
pdf_lognorm = lognorm.pdf(x, s=sigma_ln, scale=np.exp(mu_ln))
pdf_norm = norm.pdf(x, loc=mu_real, scale=sigma_real)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Görev Süresi Dağılımı: Log-Normal vs Normal', fontsize=13, fontweight='bold')

# PDF karşılaştırma
ax = axes[0]
ax.plot(x, pdf_lognorm, 'steelblue', linewidth=2.5, label='Log-Normal (seçilen)')
ax.plot(x, pdf_norm, 'tomato', linewidth=2.5, linestyle='--', label='Normal')
ax.axvline(0, color='black', linewidth=0.8, linestyle=':')
ax.fill_between(x[x<0], pdf_norm[x<0], alpha=0.3, color='tomato', label='Normal < 0 bölgesi (hatalı!)')
ax.set_xlabel('Görev süresi (s)')
ax.set_ylabel('Olasılık yoğunluğu')
ax.set_title('PDF Karşılaştırması')
ax.legend(fontsize=9)
ax.set_xlim(-0.1, 1.2)
ax.grid(alpha=0.3)

# 10,000 örnek ile empirik histogram
samples = np.random.lognormal(mu_ln, sigma_ln, 10000)
ax2 = axes[1]
ax2.hist(samples, bins=60, density=True, alpha=0.6, color='steelblue', label='Örnekler (n=10,000)')
ax2.plot(x, pdf_lognorm, 'navy', linewidth=2.5, label='Teorik PDF')
ax2.axvline(np.mean(samples), color='orange', linewidth=2, label=f'Ortalama: {np.mean(samples):.4f}s')
ax2.axvline(np.percentile(samples, 95), color='red', linewidth=2, linestyle='--', label=f'P95: {np.percentile(samples,95):.4f}s')
ax2.set_xlabel('Gunman görev süresi (s)')
ax2.set_ylabel('Yoğunluk')
ax2.set_title('Empirik Dağılım (n=10,000)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('01_lognormal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"mu_ln={mu_ln:.4f}, sigma_ln={sigma_ln:.4f}")
print(f"E[X] = {np.exp(mu_ln + sigma_ln**2/2):.4f}s (beklenen: {mu_real}s)")


## 2. Kritik Yol Yöntemi (CPM)

Her köşe bağımsız paralel olarak çalışır. Pit stop süresi en yavaş köşe tarafından belirlenir:

$$T_{pit} = T_{reaction} + \underbrace{\max_{i \in \{FL,FR,RL,RR\}}(T_i)}_{\text{kritik yol}} + T_{jack\_down}$$

Her köşe için task zinciri:
$$T_i = t_{loosen,i} + t_{tyre\_off,i} + t_{tyre\_on,i} + t_{tighten,i}$$

**Slack time** (boşluk): kritik olmayan köşelerin kritik yola göre gecikme payı:
$$S_i = T_{critical} - T_i \geq 0$$


In [ ]:
import sys
sys.path.insert(0, '..')
from pitstop.simulation.monte_carlo import run_monte_carlo, MonteCarloConfig, CORNERS, simulate_one, CREW_PROFILES

# Tek bir örnek çalıştırma ve Gantt gösterimi
import numpy as np
rng = np.random.default_rng(42)
crew = CREW_PROFILES['elite']
result = simulate_one(crew, rng=rng)

print("=== Tek Pit Stop — CPM Analizi ===")
print(f"\nKöşe süreleri:")
for corner, t in result.corner_times.items():
    slack = result.critical_path_time - t
    is_crit = corner == result.critical_corner
    marker = " ← KRİTİK YOL" if is_crit else f"  (slack: +{slack:.4f}s)"
    print(f"  {corner}: {t:.4f}s{marker}")

print(f"\nJack time: {result.jack_time:.4f}s")
print(f"Toplam pit süresi: {result.pit_time:.4f}s")

# Gantt chart
fig, ax = plt.subplots(figsize=(12, 5))
task_names = ['Lozen', 'Lastik çık.', 'Lastik tak.', 'Sıkma']
colors_normal = ['#4C9BE8', '#5BB5A2', '#E8A94C', '#E86B4C']
colors_critical = ['#C0392B', '#A93226', '#E74C3C', '#922B21']

for ci, corner in enumerate(CORNERS):
    tasks = result.task_breakdown[corner]
    start = 0
    is_crit = corner == result.critical_corner
    for ti, (task, dur) in enumerate(tasks.items()):
        color = colors_critical[ti] if is_crit else colors_normal[ti]
        ax.barh(corner, dur, left=start, height=0.5,
                color=color, edgecolor='white', linewidth=0.5, alpha=0.9)
        if dur > 0.05:
            ax.text(start + dur/2, ci, f'{dur:.3f}', ha='center', va='center',
                   fontsize=8, color='white', fontweight='bold')
        start += dur

ax.axvline(result.critical_path_time, color='red', linewidth=2, linestyle='--',
           label=f'Kritik yol: {result.critical_path_time:.4f}s')
ax.set_xlabel('Zaman (saniye)')
ax.set_title('CPM Gantt — Tek Pit Stop (Kırmızı = Kritik Yol)', fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for c, t in zip(colors_normal, task_names)]
legend_elements.append(Patch(facecolor='#C0392B', label='Kritik yol görevi'))
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('01_gantt_cpm.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Monte Carlo Engine — Tam Simülasyon

In [ ]:
# Tüm ekip profillerini karşılaştır
from pitstop.simulation.monte_carlo import compare_crews

results = {}
for crew_name in ['rookie', 'mid', 'elite', 'mercedes']:
    cfg = MonteCarloConfig(n_iterations=5000, crew_name=crew_name, seed=42)
    results[crew_name] = run_monte_carlo(cfg)

# Violin plot ile dağılım karşılaştırması
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

crew_labels = list(results.keys())
all_times = [results[c].pit_times for c in crew_labels]
colors = ['#E74C3C', '#E67E22', '#3498DB', '#27AE60']

ax = axes[0]
parts = ax.violinplot(all_times, positions=range(len(crew_labels)), showmedians=True, showextrema=True)
for i, (pc, col) in enumerate(zip(parts['bodies'], colors)):
    pc.set_facecolor(col)
    pc.set_alpha(0.7)
ax.set_xticks(range(len(crew_labels)))
ax.set_xticklabels([c.capitalize() for c in crew_labels])
ax.set_ylabel('Pit stop süresi (s)')
ax.set_title('Ekip Profilleri — Dağılım Karşılaştırması', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# KPI karşılaştırma tablosu (bar chart)
ax2 = axes[1]
metrics = {c: {'mean': results[c].mean, 'p05': results[c].p05, 'p95': results[c].p95} for c in crew_labels}
x = np.arange(len(crew_labels))
width = 0.25
bars1 = ax2.bar(x - width, [metrics[c]['p05'] for c in crew_labels], width, label='P5 (en iyi)', color='#27AE60', alpha=0.8)
bars2 = ax2.bar(x, [metrics[c]['mean'] for c in crew_labels], width, label='Ortalama', color='#3498DB', alpha=0.8)
bars3 = ax2.bar(x + width, [metrics[c]['p95'] for c in crew_labels], width, label='P95 (en kötü)', color='#E74C3C', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels([c.capitalize() for c in crew_labels])
ax2.set_ylabel('Süre (s)')
ax2.set_title('P5 / Ortalama / P95 Karşılaştırması', fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(1.4, 3.5)
plt.tight_layout()
plt.savefig('01_crew_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Özet Tablo ===")
print(compare_crews(n_iterations=3000).to_string())


## 4. Sensitivity Analizi — Hangi Görev Pit Stop'u Belirliyor?

In [ ]:
# Her görevin pit stop varyansına katkısını hesapla (partial variance)
import pandas as pd
from pitstop.simulation.monte_carlo import simulate_one, CREW_PROFILES, TASK_SEQUENCE

rng = np.random.default_rng(0)
crew = CREW_PROFILES['elite']
N = 3000

records = []
for _ in range(N):
    r = simulate_one(crew, rng=rng)
    row = {'pit_time': r.pit_time, 'critical_corner': r.critical_corner}
    for corner in ['FL','FR','RL','RR']:
        for task in TASK_SEQUENCE:
            row[f'{corner}_{task}'] = r.task_breakdown[corner][task]
    records.append(row)

df = pd.DataFrame(records)

# Pearson correlation ile sensitivity
task_cols = [f'{c}_{t}' for c in ['FL','FR','RL','RR'] for t in TASK_SEQUENCE]
correlations = {col: df['pit_time'].corr(df[col]) for col in task_cols}
corr_series = pd.Series(correlations).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#E74C3C' if v > 0.3 else '#3498DB' for v in corr_series.values]
bars = ax.barh(range(len(corr_series)), corr_series.values, color=colors, alpha=0.85)
ax.set_yticks(range(len(corr_series)))
ax.set_yticklabels([c.replace('_', ' — ') for c in corr_series.index], fontsize=9)
ax.set_xlabel("Pearson korelasyon (pit_time ile)")
ax.set_title("Sensitivity Analizi: Hangi görev pit stop süresini en çok etkiliyor?", fontweight='bold')
ax.axvline(0.3, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Yüksek etki eşiği')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('01_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nEn kritik 5 görev:")
print(corr_series.tail(5).to_string())
